In [1]:
from script.OCR import ocr
import cv2
from ultralytics import YOLO
import glob
import os
from paddleocr import PaddleOCR
import numpy as np

j:\Python\MVP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
text_ocr = ocr()

**YOLO Model (Validation Data)**

In [3]:
model = YOLO("models/best_obb.pt")

test_images = glob.glob("test_images/*.jpg") + glob.glob("test_images/*.png")
print(f"Found {len(test_images)} test images")

results = model.predict(
    source=test_images,
    save=True,
    project=r"J:\Python\MVP\outputs",
    name="predict_obb_best",
    exist_ok=True,
    conf=0.5,
    iou=0.4,
)

Found 2 test images

0: 640x480 6 0s, 1077.2ms
1: 640x480 13 0s, 1077.2ms
Speed: 545.3ms preprocess, 1077.2ms inference, 240.7ms postprocess per image at shape (1, 3, 640, 480)
Results saved to J:\Python\MVP\outputs\predict_obb_best


**Image Croping**

In [ ]:
output_dir = r"J:\Python\MVP\outputs\crops"
os.makedirs(output_dir, exist_ok=True)

for img_path, result in zip(test_images, results):
    img = cv2.imread(img_path)
    img_name = os.path.splitext(os.path.basename(img_path))[0]

    boxes = result.boxes.xyxy.cpu().numpy()
    h, w = img.shape[:2]

    for i, (x1, y1, x2, y2) in enumerate(boxes):
        x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])

        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(w, x2)
        y2 = min(h, y2)

        crop = img[y1: y2, x1: x2]

        crop_filename = os.path.join(output_dir, f"{img_name}_spine{i}.jpg")
        cv2.imwrite(crop_filename, crop)

**YOLO Model (Testing)**

In [ ]:
# model = YOLO("models/best.pt")

# test_images = glob.glob("test_images_2/*.jfif") + glob.glob("test_images_2/*.jpg")
# print(f"Found {len(test_images)} test images")

# results = model.predict(
#     source=test_images,
#     save=True,
#     project=r"G:\Python\MVP\outputs_2",
#     name="predict",
#     exist_ok=True,
#     conf=0.5
# )

**Cropping**

In [ ]:
# output_dir = r"G:\Python\MVP\outputs_2\crops"
# os.makedirs(output_dir, exist_ok=True)

# for img_path, result in zip(test_images, results):
#     img = cv2.imread(img_path)
#     img_name = os.path.splitext(os.path.basename(img_path))[0]

#     boxes = result.boxes.xyxy.cpu().numpy()
#     h, w = img.shape[:2]

#     for i, (x1, y1, x2, y2) in enumerate(boxes):
#         x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])

#         x1 = max(0, x1)
#         y1 = max(0, y1)
#         x2 = min(w, x2)
#         y2 = min(h, y2)

#         crop = img[y1: y2, x1: x2]

#         crop_filename = os.path.join(output_dir, f"{img_name}_spine{i}.jpg")
#         cv2.imwrite(crop_filename, crop)

**Gray Scaling**

In [ ]:
# os.makedirs('outputs_2/gray', exist_ok=True)
# os.makedirs('outputs_2/clahe', exist_ok=True)

# crop_paths = glob.glob('outputs_2/crops/*.jpg')
# clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

# for path in crop_paths:
#     gray = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
#     clahe_gray = clahe.apply(gray)

#     filename = os.path.basename(path)
#     cv2.imwrite(f'outputs_2/gray/{filename}', gray)
#     cv2.imwrite(f'outputs_2/clahe/{filename}', clahe_gray)

In [ ]:
# img = cv2.imread("outputs_2/crops/images_spine5.jpg")
# rotated = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
# scale = 10

# rotated = cv2.resize(
#     rotated,
#     None,
#     fx=scale,
#     fy=scale,
#     interpolation=cv2.INTER_CUBIC
# )

# # results = text_ocr.reader.readtext(rotated)
# results = text_ocr.txt(rotated)
# print(results)

In [ ]:
os.makedirs("outputs_2/preprocessed", exist_ok=True)

crop_paths = glob.glob("outputs_2/crops/*.jpg")
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

for path in crop_paths:
    img = cv2.imread(path)

    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Upscale
    scale = 3
    gray = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)

    # CLAHE
    gray = clahe.apply(gray)

    # Sharpen
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    sharpen = cv2.filter2D(gray, -1, kernel=np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]]))


    filename = os.path.basename(path)
    cv2.imwrite(os.path.join("outputs_2/preprocessed", filename), sharpen)

In [ ]:
img = cv2.imread("outputs_2/preprocessed/images (1)_spine3.jpg")
rotated = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)

results = text_ocr.txt(rotated)
print(results)

**Paddle OCR**

In [ ]:
# os.environ["FLAGS_use_mkldnn"] = "0"
# os.environ["PADDLE_PDX_ENABLE_MKLDNN_BYDEFAULT"] = "0"

In [ ]:
# Initialize PaddleOCR
paddle_ocr = PaddleOCR(
    lang='ur',
    enable_mkldnn=False,
    use_doc_orientation_classify=True,
    use_doc_unwarping=False,
    use_textline_orientation=True
)

In [ ]:
paths = glob.glob("outputs_2/crops/*.jpg")
pre_paths = glob.glob("outputs_2/preprocessed/*.jpg")

for path in paths:
    img = cv2.imread(path)
    ocr_results = paddle_ocr.predict(img)

    img_name = os.path.splitext(os.path.basename(path))[0]

    print(f'Image: {img_name}')

    for res in ocr_results:
        texts = res['rec_texts']
        scores = res['rec_scores']

        for text, score in zip(texts, scores):
            print(f'Text: {text:30} Confidence: {score:.4f}')

In [ ]:
for path in pre_paths:
    img = cv2.imread(path)
    ocr_results = paddle_ocr.predict(img)

    img_name = os.path.splitext(os.path.basename(path))[0]

    print(f'Image: {img_name}')

    for res in ocr_results:
        texts = res['rec_texts']
        scores = res['rec_scores']

        for text, score in zip(texts, scores):
            print(f'Text: {text:30} Confidence: {score:.4f}')

In [ ]:
pre_paths1 = glob.glob("outputs_2/preprocessed1/*.jpg")

for path in pre_paths1:
    img = cv2.imread(path)
    ocr_results = paddle_ocr.predict(img)

    img_name = os.path.splitext(os.path.basename(path))[0]

    print(f'Image: {img_name}')

    for res in ocr_results:
        texts = res['rec_texts']
        scores = res['rec_scores']

        for text, score in zip(texts, scores):
            print(f'Text: {text:30} Confidence: {score:.4f}')

In [ ]:
binary_path = glob.glob("outputs_2/binary/*.jpg")

for path in binary_path:
    img = cv2.imread(path)
    ocr_results = paddle_ocr.predict(img)

    img_name = os.path.splitext(os.path.basename(path))[0]

    print(f'Image: {img_name}')

    for res in ocr_results:
        texts = res['rec_texts']
        scores = res['rec_scores']

        for text, score in zip(texts, scores):
            print(f'Text: {text:30} Confidence: {score:.4f}')

In [ ]:
img = cv2.imread("outputs_2/crops/images (1)_spine7.jpg")

ocr_results = paddle_ocr.predict(img)

for res in ocr_results:
    texts = res["rec_texts"]
    scores = res["rec_scores"]

    for text, score in zip(texts, scores):
        print(f"Text: {text:30} Confidence: {score:.4f}")

In [ ]:
binary_path = glob.glob("outputs/crops/*.jpg")

for path in binary_path:
    img = cv2.imread(path)
    ocr_results = paddle_ocr.predict(img)

    img_name = os.path.splitext(os.path.basename(path))[0]

    print(f'Image: {img_name}')

    for res in ocr_results:
        texts = res['rec_texts']
        scores = res['rec_scores']

        for text, score in zip(texts, scores):
            print(f'Text: {text:30} Confidence: {score:.4f}')

In [ ]:
from script.pipeline import SpinePipeline

process = SpinePipeline()

In [ ]:
import cv2
img = cv2.imread('outputs_2/preprocessed/images (1)_spine2.jpg')
process.process(img)